# COE Quotas and Prices: A Public Data Story

This notebook prepares a reproducible analysis for Section 1 Question 2. It downloads the two official data.gov.sg CSV files, checks the shape and timing of the data, focuses on COE Categories A and B, and builds a model that can translate quota scenarios into estimated price movements.

**Story question:** when COE prices move sharply, how much of that movement is connected to available quota, and how can we explain the likely effect of adding certificates in plain public terms?


## 0. Setup

Run the next cell if the notebook environment does not already have the required packages.

In [ ]:
# Uncomment on a fresh environment.
# %pip install -q pandas numpy matplotlib seaborn statsmodels scikit-learn

In [ ]:
from __future__ import annotations

import json
import re
import subprocess
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_columns", 100)

ROOT = Path.cwd().resolve()
SECTION_DIR = ROOT.parent if ROOT.name == "codes" else ROOT
CODES_DIR = ROOT if ROOT.name == "codes" else ROOT / "codes"

DATA_DIR = CODES_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "coe_prices.csv": "d_69b3380ad7e51aff3a7dcc84eba52b8a",
    "coe_quota.csv": "d_22094bf608253d36c0c63b52d852dd6e",
}

DATA_DIR

## 1. Download Official CSVs

data.gov.sg serves dataset downloads through a short-lived signed URL. The function below asks the official API for that URL and then saves the CSV into `data/`.

In [ ]:
def download_dataset(dataset_id: str, output_path: Path) -> Path:
    api_url = f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/initiate-download"
    with urllib.request.urlopen(api_url, timeout=60) as response:
        payload = json.load(response)

    signed_url = payload["data"]["url"]
    subprocess.run(
        ["curl", "-L", "--fail", "--retry", "3", "-A", "Mozilla/5.0", signed_url, "-o", str(output_path)],
        check=True,
    )
    return output_path


for filename, dataset_id in DATASETS.items():
    path = download_dataset(dataset_id, DATA_DIR / filename)
    print(f"saved {path.relative_to(SECTION_DIR)} ({path.stat().st_size:,} bytes)")

## 2. Load and Understand the Data

The price file is already tidy at bidding-exercise level. The quota file is wider and monthly; it is useful as a cross-check and longer context. Before modelling, the notebook checks coverage by year and month so that visible breaks, thin periods, and unusual years are not hidden inside the final charts.


In [ ]:
prices = pd.read_csv(DATA_DIR / "coe_prices.csv", thousands=",")
quota_wide = pd.read_csv(DATA_DIR / "coe_quota.csv", na_values=["-"])

prices["month"] = pd.to_datetime(prices["month"] + "-01")
numeric_cols = ["bidding_no", "quota", "bids_success", "bids_received", "premium"]
prices[numeric_cols] = prices[numeric_cols].apply(pd.to_numeric, errors="coerce")

display(prices.head())
display(prices.dtypes.to_frame("dtype"))
print(f"Price data: {prices['month'].min():%Y-%m} to {prices['month'].max():%Y-%m}, {len(prices):,} rows")
print(f"Quota data: {quota_wide.shape[0]} data series x {quota_wide.shape[1] - 1} months")

In [ ]:
def parse_quota_month(label: str) -> pd.Timestamp:
    return pd.to_datetime(label, format="%Y%b")


quota_long = quota_wide.melt(id_vars="DataSeries", var_name="month_label", value_name="value")
quota_long["month"] = quota_long["month_label"].map(parse_quota_month)
quota_long["value"] = pd.to_numeric(quota_long["value"], errors="coerce")
quota_long = quota_long.dropna(subset=["value"])

quota_long.head()

### 2.1 Preprocessing Data Audit

Before moving into the model, check the basic time structure of the bidding data. This makes the analysis more transparent: we can see how many exercises appear each year, whether every calendar month is represented, and how premium, quota, and demand measures move across time.


In [ ]:
prices_audit = prices.copy()
prices_audit["year"] = prices_audit["month"].dt.year
prices_audit["calendar_month"] = prices_audit["month"].dt.month_name()

coverage_by_year = (
    prices_audit.groupby("year")
    .agg(
        first_month=("month", "min"),
        last_month=("month", "max"),
        bidding_exercises=("month", "size"),
        vehicle_classes=("vehicle_class", "nunique"),
        avg_premium=("premium", "mean"),
        total_quota=("quota", "sum"),
        total_bids_received=("bids_received", "sum"),
    )
    .assign(avg_bid_pressure=lambda df: df["total_bids_received"] / df["total_quota"])
    .round({"avg_premium": 0, "avg_bid_pressure": 2})
)

display(coverage_by_year)

month_order = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December",
]
month_by_month = (
    prices_audit.groupby("calendar_month")
    .agg(
        bidding_exercises=("month", "size"),
        avg_premium=("premium", "mean"),
        median_quota=("quota", "median"),
        avg_bids_received=("bids_received", "mean"),
    )
    .reindex(month_order)
    .round({"avg_premium": 0, "median_quota": 0, "avg_bids_received": 0})
)

display(month_by_month)


## 3. Prepare Category A and B Modeling Table

The most useful public metric is **bid pressure**: how many bids are received for each available certificate. It lets the story talk about market tightness rather than quota alone.


In [ ]:
ab = (
    prices.loc[prices["vehicle_class"].isin(["Category A", "Category B"])]
    .sort_values(["vehicle_class", "month", "bidding_no"])
    .copy()
)

ab["exercise_date"] = ab["month"] + pd.to_timedelta((ab["bidding_no"] - 1) * 14, unit="D")
ab["bid_pressure"] = ab["bids_received"] / ab["quota"]
ab["success_rate"] = ab["bids_success"] / ab["bids_received"]
ab["log_premium"] = np.log(ab["premium"])
ab["log_quota"] = np.log(ab["quota"])
ab["log_bid_pressure"] = np.log(ab["bid_pressure"])

for col in ["premium", "quota", "bid_pressure"]:
    ab[f"{col}_lag1"] = ab.groupby("vehicle_class")[col].shift(1)

ab["log_premium_lag1"] = np.log(ab["premium_lag1"])
ab["bid_pressure_roll3"] = ab.groupby("vehicle_class")["bid_pressure"].transform(lambda s: s.rolling(3, min_periods=1).mean())
ab["year"] = ab["month"].dt.year
ab["post_covid"] = (ab["month"] >= "2020-07-01").astype(int)

model_df = ab.dropna(subset=["log_premium", "log_quota", "log_bid_pressure", "log_premium_lag1"]).copy()
display(model_df.head())
model_df.groupby("vehicle_class")[["premium", "quota", "bid_pressure"]].describe().round(2)

### 3.1 Breakdowns and Outlier Review

This section checks Category A and B by year before fitting the model. The aim is to identify years that behave differently from the surrounding pattern, especially 2020, so the final story can treat them as context rather than ordinary observations.


In [ ]:
category_year_breakdown = (
    ab.groupby(["vehicle_class", "year"])
    .agg(
        exercises=("premium", "size"),
        avg_premium=("premium", "mean"),
        median_premium=("premium", "median"),
        total_quota=("quota", "sum"),
        avg_bid_pressure=("bid_pressure", "mean"),
        median_success_rate=("success_rate", "median"),
    )
    .round({
        "avg_premium": 0,
        "median_premium": 0,
        "avg_bid_pressure": 2,
        "median_success_rate": 3,
    })
)

display(category_year_breakdown)

outlier_base = ab.copy()
outlier_base["premium_yoy_pct"] = outlier_base.groupby(["vehicle_class", outlier_base["month"].dt.month])["premium"].pct_change(1)

outlier_flags = []
for metric in ["premium", "quota", "bid_pressure"]:
    q1 = outlier_base.groupby("vehicle_class")[metric].transform(lambda s: s.quantile(0.25))
    q3 = outlier_base.groupby("vehicle_class")[metric].transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    outlier_base[f"{metric}_outlier"] = (outlier_base[metric] < q1 - 1.5 * iqr) | (outlier_base[metric] > q3 + 1.5 * iqr)
    outlier_flags.append(f"{metric}_outlier")

outlier_base["any_outlier"] = outlier_base[outlier_flags].any(axis=1)

outlier_review = (
    outlier_base.loc[
        outlier_base["any_outlier"] | outlier_base["year"].eq(2020),
        [
            "month", "bidding_no", "vehicle_class", "premium", "quota",
            "bids_received", "bid_pressure", "success_rate",
            "premium_outlier", "quota_outlier", "bid_pressure_outlier",
        ],
    ]
    .sort_values(["month", "vehicle_class", "bidding_no"])
    .reset_index(drop=True)
)

display(outlier_review)

covid_context = (
    ab.assign(period=np.select(
        [
            ab["month"] < "2020-01-01",
            ab["month"].between("2020-01-01", "2020-12-31"),
            ab["month"] > "2020-12-31",
        ],
        ["Before 2020", "2020", "After 2020"],
        default="Other",
    ))
    .groupby(["vehicle_class", "period"])
    .agg(
        exercises=("premium", "size"),
        median_premium=("premium", "median"),
        median_quota=("quota", "median"),
        median_bid_pressure=("bid_pressure", "median"),
    )
    .round({"median_premium": 0, "median_quota": 0, "median_bid_pressure": 2})
)

display(covid_context)


## 4. Charts for the Public Story

The first chart keeps the supply-and-price story simple: monthly average premiums on top, monthly certificates offered below. The second chart checks whether bid pressure gives a clearer explanation than quota alone.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
for category, g in ab.groupby("vehicle_class"):
    monthly = g.groupby("month", as_index=False).agg(premium=("premium", "mean"), quota=("quota", "sum"))
    axes[0].plot(monthly["month"], monthly["premium"], label=category, linewidth=2.5)
    axes[1].plot(monthly["month"], monthly["quota"], label=category, linewidth=2.5)

axes[0].set_title("COE premiums climbed as supply became tighter")
axes[0].set_ylabel("Average premium per bidding exercise (S$)")
axes[1].set_ylabel("Monthly certificates offered")
axes[1].set_xlabel("Month")
for ax in axes:
    ax.legend()
    ax.yaxis.set_major_formatter(lambda x, pos: f"{x:,.0f}")
plt.tight_layout()

In [ ]:
plt.figure(figsize=(12, 7))
sns.scatterplot(
    data=ab, x="bid_pressure", y="premium", hue="vehicle_class", size="quota", sizes=(25, 180), alpha=0.75
)
plt.title("Premiums are highest when many bidders chase each certificate")
plt.xlabel("Bid pressure: bids received per certificate")
plt.ylabel("COE premium (S$)")
plt.gca().yaxis.set_major_formatter(lambda x, pos: f"{x:,.0f}")
plt.tight_layout()

In [ ]:
recent = ab[ab["month"] >= ab["month"].max() - pd.DateOffset(years=3)].copy()
summary = (
    recent.groupby("vehicle_class")
    .agg(
        latest_premium=("premium", "last"),
        median_premium=("premium", "median"),
        median_quota=("quota", "median"),
        median_bid_pressure=("bid_pressure", "median"),
    )
    .round(2)
)
summary

## 5. Model COE Premiums

A log-price model gives interpretable elasticities: a coefficient of `-0.10` on `log_quota` means a 1% increase in quota is associated with about a 0.10% decrease in price, holding the other included factors constant. This is association, not a randomized policy experiment.

In [ ]:
train = model_df[model_df["month"] < "2024-01-01"].copy()
test = model_df[model_df["month"] >= "2024-01-01"].copy()

formula = "log_premium ~ log_quota + log_bid_pressure + log_premium_lag1 + C(vehicle_class) + C(bidding_no) + post_covid"
model = smf.ols(formula, data=train).fit(cov_type="HC3")
print(model.summary())

In [ ]:
test = test.copy()
test["pred_premium"] = np.exp(model.predict(test))

mae = mean_absolute_error(test["premium"], test["pred_premium"])
mape = mean_absolute_percentage_error(test["premium"], test["pred_premium"])
print(f"Holdout MAE: S${mae:,.0f}")
print(f"Holdout MAPE: {mape:.1%}")

plt.figure(figsize=(10, 7))
sns.scatterplot(data=test, x="premium", y="pred_premium", hue="vehicle_class", s=90)
limit = max(test["premium"].max(), test["pred_premium"].max()) * 1.05
plt.plot([0, limit], [0, limit], color="black", linestyle="--", linewidth=1)
plt.xlim(0, limit)
plt.ylim(0, limit)
plt.title("Model check: predicted vs actual premiums since 2024")
plt.xlabel("Actual premium (S$)")
plt.ylabel("Predicted premium (S$)")
plt.gca().xaxis.set_major_formatter(lambda x, pos: f"{x:,.0f}")
plt.gca().yaxis.set_major_formatter(lambda x, pos: f"{x:,.0f}")
plt.tight_layout()

## 6. Quota Elasticity and Scenario Simulation

Because the fitted model is log-log in quota, `log_quota` is the estimated quota elasticity. The scenario table below converts that coefficient into a plain-language dollar impact from adding certificates.

In [ ]:
quota_elasticity = model.params["log_quota"]
print(f"Estimated quota elasticity: {quota_elasticity:.3f}")

latest = ab.sort_values("exercise_date").groupby("vehicle_class").tail(1).copy()
increments = [25, 50, 100, 200]

scenario_rows = []
for _, row in latest.iterrows():
    for extra in increments:
        quota_pct_change = extra / row["quota"]
        expected_pct_change = quota_elasticity * quota_pct_change
        expected_price = row["premium"] * (1 + expected_pct_change)
        scenario_rows.append(
            {
                "vehicle_class": row["vehicle_class"],
                "baseline_month": row["month"].strftime("%Y-%m"),
                "baseline_quota": int(row["quota"]),
                "baseline_premium": row["premium"],
                "extra_certificates": extra,
                "quota_change_pct": quota_pct_change,
                "estimated_price_change_pct": expected_pct_change,
                "estimated_premium_after_change": expected_price,
                "estimated_dollar_change": expected_price - row["premium"],
            }
        )

scenario = pd.DataFrame(scenario_rows)
display(
    scenario.assign(
        quota_change_pct=lambda d: (100 * d["quota_change_pct"]).round(1),
        estimated_price_change_pct=lambda d: (100 * d["estimated_price_change_pct"]).round(2),
        baseline_premium=lambda d: d["baseline_premium"].round(0).astype(int),
        estimated_premium_after_change=lambda d: d["estimated_premium_after_change"].round(0).astype(int),
        estimated_dollar_change=lambda d: d["estimated_dollar_change"].round(0).astype(int),
    )
)

## 7. Management Execution Insights

A practical management reading is that quota should be interpreted through **bid pressure**, not in isolation. This section adds execution-ready views for policy discussion and public communication:

1. direct quota effect versus fixed-demand quota effect;
2. scenarios that vary both quota and demand;
3. simple uncertainty ranges using holdout model error;
4. category and regime differences; and
5. a clear causality caveat.


### 7.1 Direct Quota Effect vs Fixed-Demand Quota Effect

The fitted model includes both `log_quota` and `log_bid_pressure`. Since `bid_pressure = bids_received / quota`, quota enters the model in two ways.

- **Controlled interpretation:** change quota while holding bid pressure constant. This is the direct `log_quota` coefficient.
- **Fixed-demand interpretation:** add certificates while bids received remain unchanged. In this case, bid pressure falls, so the quota impact combines `log_quota` and `log_bid_pressure`.

The fixed-demand interpretation is usually more intuitive for management scenarios and public explanation.

In [ ]:
direct_quota_elasticity = model.params["log_quota"]
bid_pressure_elasticity = model.params["log_bid_pressure"]
fixed_demand_quota_elasticity = direct_quota_elasticity - bid_pressure_elasticity

elasticity_summary = pd.DataFrame(
    [
        {
            "interpretation": "Controlled quota effect",
            "what_changes": "Quota changes; bid pressure held constant",
            "elasticity": direct_quota_elasticity,
        },
        {
            "interpretation": "Fixed-demand quota effect",
            "what_changes": "Quota changes; bids received unchanged, so bid pressure changes",
            "elasticity": fixed_demand_quota_elasticity,
        },
    ]
)
elasticity_summary.assign(elasticity=lambda d: d["elasticity"].round(3))

### 7.2 Scenario Matrix: Quota Change, Demand Response, and Uncertainty

Point estimates can create false precision. The scenario matrix below varies quota and demand together, then adds a simple uncertainty range using the holdout MAE. This is not a formal prediction interval, but it is more decision-ready than a single number.

In [ ]:
holdout_mae = mae
q_coef = model.params["log_quota"]
bp_coef = model.params["log_bid_pressure"]
latest = ab.sort_values("exercise_date").groupby("vehicle_class").tail(1).copy()

scenario_rows = []
for _, row in latest.iterrows():
    for quota_change in [0.05, 0.10, 0.20]:
        for demand_change in [0.00, 0.05, 0.10]:
            log_price_change = (
                q_coef * np.log1p(quota_change)
                + bp_coef * (np.log1p(demand_change) - np.log1p(quota_change))
            )
            estimated_premium = row["premium"] * np.exp(log_price_change)
            scenario_rows.append(
                {
                    "vehicle_class": row["vehicle_class"],
                    "baseline_month": row["month"].strftime("%Y-%m"),
                    "baseline_premium": row["premium"],
                    "quota_change_pct": 100 * quota_change,
                    "demand_change_pct": 100 * demand_change,
                    "estimated_price_change_pct": 100 * (np.exp(log_price_change) - 1),
                    "estimated_premium": estimated_premium,
                    "low_using_holdout_mae": estimated_premium - holdout_mae,
                    "high_using_holdout_mae": estimated_premium + holdout_mae,
                }
            )

scenario_matrix = pd.DataFrame(scenario_rows)
display(
    scenario_matrix.assign(
        baseline_premium=lambda d: d["baseline_premium"].round(0).astype(int),
        quota_change_pct=lambda d: d["quota_change_pct"].round(1),
        demand_change_pct=lambda d: d["demand_change_pct"].round(1),
        estimated_price_change_pct=lambda d: d["estimated_price_change_pct"].round(2),
        estimated_premium=lambda d: d["estimated_premium"].round(0).astype(int),
        low_using_holdout_mae=lambda d: d["low_using_holdout_mae"].round(0).astype(int),
        high_using_holdout_mae=lambda d: d["high_using_holdout_mae"].round(0).astype(int),
    )
)

### 7.3 Category and Regime Differences

Category A and Category B can respond differently because they serve different buyer groups and have different baseline quotas. Market behaviour may also differ before and after COVID. Segment-level elasticities should be treated as directional because each segment has fewer observations than the pooled model.

In [ ]:
def fit_segment_ols(df: pd.DataFrame, features: list[str]) -> tuple[pd.Series, float, int]:
    segment_model = smf.ols(
        "log_premium ~ " + " + ".join(features),
        data=df,
    ).fit(cov_type="HC3")
    return segment_model.params, segment_model.rsquared, int(segment_model.nobs)

segment_specs = [
    ("Category A only", model_df[(model_df["month"] < "2024-01-01") & (model_df["vehicle_class"] == "Category A")], ["log_quota", "log_bid_pressure", "log_premium_lag1", "C(bidding_no)", "post_covid"]),
    ("Category B only", model_df[(model_df["month"] < "2024-01-01") & (model_df["vehicle_class"] == "Category B")], ["log_quota", "log_bid_pressure", "log_premium_lag1", "C(bidding_no)", "post_covid"]),
    ("Pre-COVID regime", model_df[model_df["month"] < "2020-07-01"], ["log_quota", "log_bid_pressure", "log_premium_lag1", "C(vehicle_class)", "C(bidding_no)"]),
    ("Post-COVID regime", model_df[(model_df["month"] >= "2020-07-01") & (model_df["month"] < "2024-01-01")], ["log_quota", "log_bid_pressure", "log_premium_lag1", "C(vehicle_class)", "C(bidding_no)"]),
]

segment_rows = []
for segment, segment_df, features in segment_specs:
    params, r2_segment, nobs = fit_segment_ols(segment_df, features)
    segment_rows.append(
        {
            "segment": segment,
            "rows": nobs,
            "direct_quota_elasticity": params["log_quota"],
            "bid_pressure_elasticity": params["log_bid_pressure"],
            "fixed_demand_quota_elasticity": params["log_quota"] - params["log_bid_pressure"],
            "r_squared": r2_segment,
        }
    )

segment_elasticity = pd.DataFrame(segment_rows)
display(segment_elasticity.round(3))

### 7.4 Execution and Communication Takeaways

For management execution:

- monitor bid pressure as the primary market-tightness signal;
- use quota scenarios together with demand-response scenarios;
- present ranges, not just point estimates;
- review Category A and B separately before making category-specific decisions; and
- refresh the model after each quota review cycle.

For public perception:

- explain that COE premiums reflect how crowded each bidding exercise is;
- describe quota as one lever that changes market pressure, rather than a price switch; and
- avoid promising exact price reductions from adding certificates.

**Causality caveat:** these estimates are historical associations from bidding data. They are useful for scenario planning, but they do not prove that quota changes alone will mechanically produce the estimated price changes.


## 8. Public-Facing Data Story Outline

1. **Opening:** COE prices feel volatile because every bidding exercise is a small market where limited certificates meet changing demand.
2. **Preprocessing check:** show the year-by-year and month-by-month coverage so the reader can see where the data is dense, thin, or unusual.
3. **Chart 1:** show Category A and B premiums against monthly quota. The public takeaway is that the supply cycle matters, but it is not the whole story.
4. **Chart 2:** show bid pressure. This reframes the issue: prices spike when many bidders chase each available certificate.
5. **Outlier note:** call out years such as 2020 separately, because policy restrictions and pandemic-era demand changes can distort normal patterns.
6. **Model result:** explain elasticity in one sentence: based on historical bidding exercises, a 1% increase in quota is associated with an estimated `quota_elasticity`% movement in premiums, after accounting for recent price levels and bid pressure.
7. **Scenario table:** translate policy levers into concrete examples, such as adding 50 or 100 certificates to Category A or B.
8. **Caveat:** the model estimates historical association, not a guaranteed causal effect. Dealer expectations, economic conditions, car prices, loan rules, and Category E substitution can also move premiums.
